# Few-shot baselines

This notebook runs the few-shot baseline of Qwen3-1.7B and Qwen3-4B on all splits and prints the gate for each. It has no logic of its own: every step calls `make`.

On Kaggle, use one T4 and turn Internet on. With `DRY_RUN = True` the notebook runs without a GPU. It skips the install and the clone, uses the checkout it is in, starts a fake server instead of vLLM, and writes to a temporary folder. A dry run uses a fake model, so its results say nothing about a real model. Never copy them into `results/`.

In [ ]:
DRY_RUN = False
REPO_URL = "https://github.com/mgetech/open-llm-adaptation.git"
REPO_REF = "main"  # a tag or a commit
MODELS = ["qwen3-1.7b", "qwen3-4b"]

DRY = ["DRY_RUN=1"] if DRY_RUN else []
OFFLINE = ["OFFLINE=1"] if DRY_RUN else []

In [ ]:
import os
import subprocess
import tempfile
from pathlib import Path


def run(*command, check=True):
    options = dict(stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with subprocess.Popen(command, encoding="utf-8", errors="replace", **options) as process:
        for line in process.stdout:
            print(line, end="")
    if check and process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)


def make(*arguments, check=True):
    run("make", *arguments, check=check)


if DRY_RUN:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    OUT_DIR = Path(tempfile.mkdtemp(prefix="baselines-dry-run-")) / "eval"
else:
    REPO = Path("/kaggle/working/open-llm-adaptation")
    OUT_DIR = Path("/kaggle/working/results/eval")
print("repository:", REPO)
print("results:", OUT_DIR)

## Python 3.12

Kaggle's default Python may be older, so uv installs its own 3.12.

In [ ]:
if DRY_RUN:
    print("dry run: skipped")
else:
    run("pip", "install", "--quiet", "uv")
    run("uv", "python", "install", "3.12")

## Repository

The data is in the repository, so nothing else is downloaded.

In [ ]:
if DRY_RUN:
    print("dry run: skipped")
else:
    run("git", "clone", REPO_URL, str(REPO))
    run("git", "-C", str(REPO), "checkout", REPO_REF)
os.chdir(REPO)

## Project and vLLM

The project is installed from the lock file. vLLM is the version pinned in `configs/models.yaml`.

In [ ]:
if DRY_RUN:
    print("dry run: skipped")
else:
    make("setup")
    make("install-vllm")

## Model checks

The licence, size and serving checks, and the pinned revisions against the Hub.

In [ ]:
make("models-check", *OFFLINE)

## Baselines

For each model, `make baseline` starts the server, runs dev, test_iid, test_ood, edge and gold with the few-shot prompt, and stops the server. Loading the model takes a few minutes the first time.

In [ ]:
for model in MODELS:
    make("baseline", f"MODEL={model}", f"OUT_DIR={OUT_DIR.as_posix()}", *DRY)

## Gate

A baseline is expected to fail the gate. The table shows by how much.

In [ ]:
for model in MODELS:
    make("gate", f"SYSTEM={model}-fewshot", f"OUT_DIR={OUT_DIR.as_posix()}", check=False)

## Results

Copy the folder `tax-booking` from the results folder into `results/eval/` of the repository.

In [ ]:
for path in sorted(OUT_DIR.rglob("*.json")):
    print(path)